In [1]:
%reload_ext autoreload
%autoreload 2
%cd ../../

/home/hazzu/Code/thesis


In [2]:
import os
import random
import shutil


def split_dataset_by_identity_and_image_ratio(
    source_dir, train_dir, val_dir, test_dir, identity_ratio=(0.5, 0.4, 0.1)
):
    """
    Chia tập dữ liệu theo danh tính và cố gắng tuân theo tỷ lệ ảnh.

    Args:
        source_dir (str): Đường dẫn đến thư mục gốc chứa các thư mục danh tính.
        train_dir (str): Đường dẫn đến thư mục lưu trữ tập huấn luyện.
        val_dir (str): Đường dẫn đến thư mục lưu trữ tập validation.
        test_dir (str): Đường dẫn đến thư mục lưu trữ tập kiểm tra.
        identity_ratio (tuple): Tỷ lệ (train, val, test) cho số lượng danh tính.
    """

    # Tạo các thư mục đích nếu chưa tồn tại
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(val_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)

    all_identities = [
        d for d in os.listdir(source_dir) if os.path.isdir(os.path.join(source_dir, d))
    ]
    random.shuffle(all_identities)  # Xáo trộn danh sách danh tính

    num_identities = len(all_identities)
    num_train_identities = int(num_identities * identity_ratio[0])
    num_val_identities = int(num_identities * identity_ratio[1])
    # num_test_identities còn lại

    train_identities = all_identities[:num_train_identities]
    val_identities = all_identities[
        num_train_identities : num_train_identities + num_val_identities
    ]
    test_identities = all_identities[num_train_identities + num_val_identities :]

    print(f"Tổng số danh tính: {num_identities}")
    print(
        f"Số danh tính cho Train: {len(train_identities)} (Mong muốn: {num_train_identities})"
    )
    print(
        f"Số danh tính cho Validation: {len(val_identities)} (Mong muốn: {num_val_identities})"
    )
    print(f"Số danh tính cho Test: {len(test_identities)}")

    identity_counts = {"train": 0, "val": 0, "test": 0}
    image_counts = {"train": 0, "val": 0, "test": 0}
    total_images_original = 0

    # Hàm để sao chép ảnh và đếm
    def copy_and_count(identities, target_base_dir, set_name):
        for identity_folder in identities:
            source_identity_path = os.path.join(source_dir, identity_folder)
            target_identity_path = os.path.join(target_base_dir, identity_folder)
            os.makedirs(target_identity_path, exist_ok=True)

            identity_counts[set_name] += 1
            num_images_in_identity = 0
            for image_file in os.listdir(source_identity_path):
                if image_file.lower().endswith((".png", ".jpg", ".jpeg")):
                    source_image_path = os.path.join(source_identity_path, image_file)
                    target_image_path = os.path.join(target_identity_path, image_file)
                    shutil.copy2(source_image_path, target_image_path)
                    image_counts[set_name] += 1
                    num_images_in_identity += 1
            # print(f"  Danh tính {identity_folder} ({num_images_in_identity} ảnh) được gán cho {set_name}")

    # Đếm tổng số ảnh ban đầu
    for identity_folder in all_identities:
        source_identity_path = os.path.join(source_dir, identity_folder)
        for image_file in os.listdir(source_identity_path):
            if image_file.lower().endswith((".png", ".jpg", ".jpeg")):
                total_images_original += 1
    print(f"\nTổng số ảnh ban đầu: {total_images_original}")

    print("\nBắt đầu sao chép tập Train...")
    copy_and_count(train_identities, train_dir, "train")
    print("Hoàn thành tập Train.")

    print("\nBắt đầu sao chép tập Validation...")
    copy_and_count(val_identities, val_dir, "val")
    print("Hoàn thành tập Validation.")

    print("\nBắt đầu sao chép tập Test...")
    copy_and_count(test_identities, test_dir, "test")
    print("Hoàn thành tập Test.")

    print("\n--- Thống kê số lượng ---")
    print(
        f"Tập Train: {identity_counts['train']} danh tính, {image_counts['train']} ảnh"
    )
    print(
        f"Tập Validation: {identity_counts['val']} danh tính, {image_counts['val']} ảnh"
    )
    print(f"Tập Test: {identity_counts['test']} danh tính, {image_counts['test']} ảnh")

    total_images_split = (
        image_counts["train"] + image_counts["val"] + image_counts["test"]
    )
    print(
        f"Tổng số ảnh đã chia: {total_images_split} (Kiểm tra: {total_images_split == total_images_original})"
    )

    # Tính toán tỷ lệ ảnh thực tế
    if total_images_split > 0:
        train_image_ratio_actual = image_counts["train"] / total_images_split
        val_image_ratio_actual = image_counts["val"] / total_images_split
        test_image_ratio_actual = image_counts["test"] / total_images_split
        print("\n--- Tỷ lệ ảnh thực tế ---")
        print(
            f"Train: {train_image_ratio_actual:.2f} (Mong muốn: {identity_ratio[0]:.2f})"
        )
        print(
            f"Validation: {val_image_ratio_actual:.2f} (Mong muốn: {identity_ratio[1]:.2f})"
        )
        print(
            f"Test: {test_image_ratio_actual:.2f} (Mong muốn: {identity_ratio[2]:.2f})"
        )

In [3]:
SOURCE_DATA_DIR = "datasets/selfies"
TRAIN_DATA_DIR = "datasets/ver1/selfies_1"
VAL_DATA_DIR = "datasets/ver1/selfies_2"
TEST_DATA_DIR = "datasets/ver1/selfies_3"

identity_split_ratio = (0.5, 0.5, 0.0)

if not os.path.isdir(SOURCE_DATA_DIR):
    print(f"Lỗi: Thư mục nguồn '{SOURCE_DATA_DIR}' không tồn tại.")
else:
    split_dataset_by_identity_and_image_ratio(
        SOURCE_DATA_DIR,
        TRAIN_DATA_DIR,
        VAL_DATA_DIR,
        TEST_DATA_DIR,
        identity_split_ratio,
    )

Tổng số danh tính: 140
Số danh tính cho Train: 70 (Mong muốn: 70)
Số danh tính cho Validation: 70 (Mong muốn: 70)
Số danh tính cho Test: 0

Tổng số ảnh ban đầu: 541

Bắt đầu sao chép tập Train...
Hoàn thành tập Train.

Bắt đầu sao chép tập Validation...
Hoàn thành tập Validation.

Bắt đầu sao chép tập Test...
Hoàn thành tập Test.

--- Thống kê số lượng ---
Tập Train: 70 danh tính, 277 ảnh
Tập Validation: 70 danh tính, 264 ảnh
Tập Test: 0 danh tính, 0 ảnh
Tổng số ảnh đã chia: 541 (Kiểm tra: True)

--- Tỷ lệ ảnh thực tế ---
Train: 0.51 (Mong muốn: 0.50)
Validation: 0.49 (Mong muốn: 0.50)
Test: 0.00 (Mong muốn: 0.00)
